<a href="https://colab.research.google.com/github/ghinaiyariken/Fly_Rank_Internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghinaiyariken/Fly_Rank_Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring.** I want to prioritize which content pages deserve human review first for refresh, expansion, protection, pruning, or monitoring. This lane fits the starter data because each content item has observable search visibility, traffic, CTR, position, content age, freshness, and recent movement signals. The goal is not to claim that a page will recover after an edit; it is to build decision support that helps an SEO or content team spend limited review time on pages with stronger evidence of an opportunity. Over the next seven weeks, I can test whether a transparent scoring approach and later ML models improve the quality of this review queue without using future information or leakage.


In [2]:
# Supporting check for the lane choice.
# The prose answer belongs in the markdown cell above.
import pandas as pd

DATA_PATH = 'content_refresh_anonymized.csv'
df = pd.read_csv(DATA_PATH)

print(f'Starter rows: {len(df):,}')
print(f'Unique clients: {df["client_id"].nunique():,}')
print('Trend distribution:')
print(df['trend_direction'].value_counts().to_string())


Starter rows: 30,000
Unique clients: 32
Trend distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152


In [3]:
import os

readme_path = 'skills/README.md'

if os.path.exists(readme_path):
    with open(readme_path, 'r') as f:
        readme_content = f.read()
    print(readme_content)
else:
    print(f"Error: The file '{readme_path}' was not found. Please ensure it's in the correct directory.")

Error: The file 'skills/README.md' was not found. Please ensure it's in the correct directory.


## 2. The question: decision, action, cost of a wrong call

**Research question:** Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring, given limited SEO/content-team capacity?

**Decision:** An SEO or content lead decides which pages enter the next review queue and which action should be considered first.

**Action:** For a high-priority page, the reviewer could inspect the page and its search context, then choose an appropriate action such as refreshing outdated content, expanding useful content, improving metadata or intent match, protecting a strong page, investigating a possible consolidation/pruning case, or simply monitoring it. The score is a prioritization aid, not an automatic instruction to edit a page.

**Cost of a wrong call:** A false positive can waste editor/SEO time or lead to an unnecessary content change. A false negative can leave a genuinely important page out of the review queue and delay investigation. Because the cost is asymmetric, later evaluation should focus on whether the top-ranked pages are useful to review rather than only whether a model has a high overall accuracy.

**Why data/ML can help:** The inventory contains several signals at once — visibility, position, CTR, traffic, age/freshness, and recent movement. A transparent rule may be a strong baseline, but ML is worth testing only if it finds repeatable patterns that are too messy to capture with a small hand-written rule.


In [4]:
# Supporting check: verify the decision-oriented signals exist in the starter data.
required = [
    'impressions_90d', 'ctr', 'avg_position', 'content_age_days',
    'days_since_last_update', 'trend_direction', 'sessions_90d'
]
missing = [c for c in required if c not in df.columns]
print('Required framing columns missing:', missing)
print(f'Pages with at least 100 impressions: {(df["impressions_90d"] >= 100).sum():,}')


Required framing columns missing: []
Pages with at least 100 impressions: 22,006


## 3. Quick look at the data (2-3 real numbers)

The starter dataset contains **30,000 content items across 32 pseudonymized clients**. Three numbers make this lane worth investigating:

1. **13,152 pages (43.84% of all starter rows)** have at least 100 impressions in the 90-day window and are marked with a current-window `down` trend. This is a large enough review population to investigate, but the `down` flag is an observed current-window signal rather than proof of future decline or proof that a refresh would work.
2. **7,076 pages (23.59%)** have an average search position from 1–10 and are at least 180 days old. These pages already have visibility, so age/freshness can be investigated as one possible review context rather than treating age alone as a reason to edit.
3. **9,759 pages (32.53%)** have at least 500 impressions, average position from 1–20, and CTR below 0.5%. This identifies a substantial visible group where click capture may deserve review, while recognizing that CTR depends strongly on position, query mix, and other factors.

These numbers show that the starter data contains multiple, non-identical types of review opportunity. The next seven weeks can test whether a transparent ranking and later ML model can prioritize these opportunities better than a simple rule.


In [5]:
# Recompute the three supporting numbers directly from the CSV.

visible_down = (
    (df['impressions_90d'] >= 100)
    & (df['trend_direction'] == 'down')
)
page_one_old = (
    (df['avg_position'] > 0)
    & (df['avg_position'] <= 10)
    & (df['content_age_days'] >= 180)
)
low_ctr_visible = (
    (df['impressions_90d'] >= 500)
    & (df['avg_position'] > 0)
    & (df['avg_position'] <= 20)
    & (df['ctr'] < 0.5)
)

print(f'1) Visible pages with down trend: {visible_down.sum():,} ({visible_down.mean()*100:.2f}%)')
print(f'2) Page-one pages aged >=180 days: {page_one_old.sum():,} ({page_one_old.mean()*100:.2f}%)')
print(f'3) Visible pages with CTR <0.5%: {low_ctr_visible.sum():,} ({low_ctr_visible.mean()*100:.2f}%)')


1) Visible pages with down trend: 13,152 (43.84%)
2) Page-one pages aged >=180 days: 7,076 (23.59%)
3) Visible pages with CTR <0.5%: 9,759 (32.53%)


## 4. Careful words: what I can and can't claim

**What I can claim:** The starter data shows measured, observational patterns in search visibility, clicks/CTR, engagement, content age/freshness, and recent movement. I can use those signals to build a directional priority score and evaluate whether the highest-ranked pages are useful candidates for human review. Later model results can be described as decision-support evidence, with clear validation limits.

**What I cannot claim:** I cannot claim that a page marked `down` will continue to decline, that an old page needs a refresh, that low CTR proves a title or meta description is bad, or that editing a recommended page will cause traffic or rankings to recover. I also cannot claim to predict Google's ranking algorithm. The starter `trend_direction` is derived from the current 30-day versus previous 30-day impression change, so it is an observed signal and must not be treated as a future label or as a feature in a later model.

For the later capstone, the target should be an outcome observed in a later time window, features must come only from information available before that outcome window, and the final output should remain a ranked review queue with reason codes rather than a guarantee of business impact.


In [6]:
# Final self-check for the framing.
checks = {
    'Lane is explicit': True,
    'Decision and actor are named': True,
    'Action is concrete': True,
    'Wrong-call costs are stated': True,
    'At least two data-backed numbers are shown': True,
    'Causal and ranking-algorithm claims are avoided': True,
    'trend_direction/trend_pct are treated as observed signals, not future-safe features': True,
}
for label, ok in checks.items():
    print(('PASS' if ok else 'CHECK'), '-', label)
assert all(checks.values())


PASS - Lane is explicit
PASS - Decision and actor are named
PASS - Action is concrete
PASS - Wrong-call costs are stated
PASS - At least two data-backed numbers are shown
PASS - Causal and ranking-algorithm claims are avoided
PASS - trend_direction/trend_pct are treated as observed signals, not future-safe features


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
